# PRIMs Tutorial — Reference

This notebook collects the reference material of the tutorial: the model parameters, the script language and its functions, batch files, the Python form of a model, and the notebook functions of pyprims. Run the cells at the end to check things against your installed version of pyprims.

**Contents**

1. Parameters
2. Script syntax
3. Script functions
4. Batch files
5. Models in Python
6. The notebook session (`Session`)

## 1. Parameters

Parameters are set in the task definition (`define task name { ... }`), with a colon after the name, or from the script with `sgp("name:", value)`. `t` is true and `nil` is false. In the Python form of a model, `-` becomes `_` and the colon is dropped (`default-operator-self-assoc:` becomes `default_operator_self_assoc=`).

### Declarative memory

| parameter | meaning | default |
|---|---|---|
| `bll:` | base-level decay $d$ | 0.5 |
| `ol:` | optimized learning; `nil` for the standard equation | t |
| `mas:` | maximum associative strength $S$ | 3.0 |
| `rt:` | retrieval threshold $\tau$ | −2.0 |
| `lf:` | latency factor $F$ | 0.2 |
| `ans:` | activation noise | 0.2 |
| `mp:` | mismatch penalty | 5.0 |
| `pm:` | use partial matching to retrieve facts | nil |
| `blending:` | use blending to retrieve facts | nil |
| `ga:` | spreading activation $W$ from the goal | 1.0 |
| `input-activation:` | spreading activation $W$ from the input | 0.0 |
| `retrieval-activation:` | spreading activation $W$ from the retrieval buffer | 0.0 |
| `imaginal-activation:` or `wm-activation:` | spreading activation $W$ from working memory | 0.0 |
| `goal-chunk-spreads:` | spreading from the goal equals the activation of the goal chunk, instead of `ga:` divided by the number of chunks in the goal; this lets you give goals more or less priority | nil |
| `default-activation:` | lower bound on the base-level activation of all chunks defined in the model (including operators); `nil` lets all chunks decay | 0.0 |
| `retrieval-reinforces:` | a retrieval adds a reference to the chunk, as in standard ACT-R | nil |

### Operators

| parameter | meaning | default |
|---|---|---|
| `default-operator-assoc:` | $S_{ji}$ between an operator and the skill it is defined in; also $\max S_{ji}$ in operator learning | 3.0 |
| `default-inter-operator-assoc:` | $S_{ji}$ between operators of the same skill | 1.0 |
| `default-operator-self-assoc:` | $S_{ji}$ between an operator and itself; negative, so that an operator is less likely to fire repeatedly | −1.0 |
| `skill-operator-learning:` (or `goal-operator-learning:`) | learn the associations between the skills in the goal buffer and the operators, by reinforcement learning (Unit 3) | nil |
| `operator-bll:` | an operator that led to a reward gets an extra reference (needs skill-operator learning) | nil |
| `inter-operator-learning:` | also learn associations between operators; they start at 0 and converge to `default-inter-operator-assoc:` | nil |
| `context-operator-learning:` | learn associations between any chunk in any buffer slot and the operator | nil |
| `beta:` | learning rate $\beta$ of operator learning | 0.1 |
| `reward:` | reward for operator learning, in seconds; also the maximum time the model tries to reach the goal. 0 switches operator learning off | 0.0 |
| `explore-exploit:` | extra noise on skill–operator combinations that have not been tried often; higher values mean longer exploration (experimental) | 0.0 |
| `opc:` | operator compilation: compile pairs of operators into new operators on reward (experimental, Unit 3) | nil |
| `bindings-in-dm:` | bindings are stored in and retrieved from declarative memory, with time costs and possible failure (Unit 2) | nil |

### Productions and timing

| parameter | meaning | default |
|---|---|---|
| `egs:` | utility noise | 0.05 |
| `alpha:` | learning rate $\alpha$ of production utilities | 0.1 |
| `procedural-reward:` | reward for production utility learning | 4.0 |
| `nu:` | utility of a newly compiled production | 0.0 |
| `primU:` | utility of the productions that carry out a single PRIM | 2.0 |
| `dat:` | time to fire the first production of an operator | 0.05 |
| `production-prim-latency:` | time to fire each further production of an operator. This includes the time to retrieve the PRIM, so a fully compiled operator takes only `dat:` | 0.3 |
| `imaginal-delay:` | time to put a new chunk in working memory | 0.2 |
| `perception-action-latency:` | time of an action that is not defined with `define action` | 0.2 |
| `time-threshold:` | maximum duration of one trial in simulated seconds, to stop endless loops | 200 |
| `time-t0:`, `time-a:`, `time-b:` | parameters of ACT-R's temporal module | 0.011, 1.1, 0.015 |

The cell below lists the parameter names that your version of pyprims accepts:

In [1]:
from pyprims.model import PARAMETER_NAMES
print(" ".join(PARAMETER_NAMES))

alpha: ans: batch-trace: beta: bindings-in-dm: blending: bll: complex-goal-variable-structure: context-operator-learning: dat: declarative-buffer-stuffing: default-activation: default-inter-operator-assoc: default-operator-assoc: default-operator-self-assoc: egs: explore-exploit: extended-associations: ga: goal-chunk-spreads: goal-operator-learning: imaginal-activation: imaginal-delay: input-activation: inter-operator-learning: lf: mas: mp: new-pm-exp: new-pm-pow: nu: ol: opc: operator-bll: perception-action-latency: pm: primU: procedural-reward: production-prim-latency: retrieval-activation: retrieval-reinforces: reward: rt: skill-operator-learning: time-a: time-b: time-t0: time-threshold: wm-activation:


## 2. Script syntax

The script language has integers, reals, strings and arrays. Arrays can hold items of different types. Types are implicit. There is no Boolean type: 0 is false, and all other integers are true.

A script consists of a sequence of statements: assignments, `while` loops, `for` loops, `if` statements and function calls.

**Assignment** assigns a value to a variable, or to an element of an array:

```
a = (10 + 5) * 8
b = ["letter", 8, 4.5, random(10)]
b[2] = 10
b[3] = [1, 1, 2, 3, 5, 8]
```

Arrays start at index 0. An assignment to an element beyond the end of an array extends the array. This loop creates an array of 5 random numbers:

```
a = []
for i in 0 to 4 {
    a[i] = random(10)
}
```

**If, while and for**

```
if (condition) { then-statements } else { else-statements }
while (condition) { statements }
for index in startindex to endindex { statements }     // both ends inclusive
for item in array { statements }
```

The `else` part is optional, but parentheses and braces are required. Conditions combine comparisons with `&&` (and), `||` (or) and `!` (not).

**Things to watch out for:**

* Each comparison in a combination needs its own parentheses: `((a == 1) && (b == 2))`.
* Binary operators group to the right: `a - b - c` means `a - (b - c)`.
* Names may contain `-` and `+`, so write `i + 1`, not `i+1` (which is a single name).
* Division of two integers truncates.
* `+` with a string on the left concatenates.
* A variable first assigned inside an `if`, `while` or `for` block stays visible after the block.

**Script types.** Each model has a main script (`define script`) that is run for each trial. Optionally, a model has an initialization script (`define init-script`) that is run once, when the model is loaded. This can be useful to add larger amounts of knowledge to declarative memory. For example, an alternative way to add the count facts to the counting model:

```
define init-script {
    print("Init script: Adding count-facts to memory")
    digits = ["one","two","three","four","five","six","seven","eight","nine","ten"]
    for i in 0 to 8 {
        name = random-string("count-fact")
        add-dm(name, "count-fact", digits[i], digits[i + 1])
        set-activation(name, 1.0)
    }
}
```

## 3. Script functions

In the Python form of a model, every function is a method of `m`, with `-` written as `_`: `run-until-action("say")` becomes `m.run_until_action("say")`.

### Running the model

| function | meaning |
|---|---|
| `run-step()` | run the model for one operator |
| `run-until-action(values)` | run until the model takes a particular action. Only the slots given are compared: `run-until-action("say")` stops at any say action; `run-until-action()` at any action |
| `run-relative-time(time)` | run the model for a number of seconds |
| `run-absolute-time(time)` | run the model until a particular time (`time()` gives the current time) |
| `run-relative-time-or-action(time, values)` | run until the action is taken, or the time has passed (also `run-until-relative-time-or-action`) |
| `run-absolute-time-or-action(time, values)` | the same, until a particular time |

### Perception and action

| function | meaning |
|---|---|
| `screen(items)` | put the items in the input buffer: the first in V1, the second in V2, and so on. The argument can also be a single chunk from `define visual` (Unit 4). Also `set-input` |
| `add-visual(name, values)` | add a chunk to the visual store from the script |
| `last-action()` | an array with the last action (AC1 first, then AC2, ...). If there was no action, an array with a single empty string |

### Trials, rewards and data

| function | meaning |
|---|---|
| `trial-start()` | set the start time of the trial to the current time. Only needed when one script run has several trials |
| `trial-end()` | end of a trial: prepares the model for the next trial, and writes a line to the batch output. Always include it |
| `issue-reward(number)` | reward for operator learning; without an argument, the `reward:` parameter |
| `sleep(time)` | move the model time forward without running the model (affects decay) |
| `data-line(values)` | write a `data-line` event with up to three values to the batch output |
| `set-data-file-field(n, value)` | put a value in extra output column `in0` … `in4` (n = 0 … 4), e.g., the condition |
| `batch-parameters()` | an array with the extra arguments of the `run` command in the batch file; `"NA"` when not running in batch |
| `plot-point(number)` | plot a custom value for this trial (accuracy, for example) instead of the trial time |
| `set-graph-title(string)` | title of the results graph |
| `set-average-window(number)` | smooth the results graph with a moving average over this many trials |
| `print(values)` | print to the trace (and, with `Session(echo_print=True)`, to the notebook) |

### Changing and inspecting the model

| function | meaning |
|---|---|
| `time()` | the current model time |
| `add-dm(name, values)` | add a chunk to declarative memory; the values go into slot1 … slotn. Use `random-string` to make names |
| `set-activation(name, value)` | set the lower bound (default activation) of a chunk |
| `get-activation(name)` | the current activation of a chunk |
| `set-references(name, n)` | set the number of references of a chunk |
| `set-sji(chunkj, chunki, value)` | set the $S_{ji}$ between two chunks |
| `sgp(parameter, value)` | set a parameter, e.g., `sgp("lf:", 0.3)` |
| `add-binding(variable, value)` | bind a skill variable (`*variable`) to a value (Unit 2) |
| `purge-bindings()` | remove all bindings from declarative memory (with `bindings-in-dm: t`) |
| `set-goal(skill, [slot, value], ...)` | set the slots of a skill chunk; also `set-skill` |
| `create-new-goal(name, [slot, value], ...)` | create a new skill chunk and put it in G1; also `create-new-skill` |
| `instantiate-skill(skill, name, [slot, value], ...)` | create an instantiation of a skill |
| `imaginal-to-dm()` | move the working memory chunk to declarative memory |
| `set-buffer-slot(buffer, slot, value)` | set a buffer slot, e.g., `set-buffer-slot("input", "slot1", "nine")` |
| `get-buffer-slot(buffer, slot)` | the value of a buffer slot |

### Other

| function | meaning |
|---|---|
| `random(n)` | a random integer from 0 to n − 1 |
| `shuffle(array)` | the array in random order |
| `length(array)` | the number of elements |
| `int(x)`, `str-to-int(string)` | convert to an integer |
| `random-string(prefix)` | a unique name starting with the prefix (default `fact`) |
| `read-file(filename)` | read a file from the model's folder: an array with its non-empty lines |

## 4. Batch files

A batch file (`.bprims`) runs one or more models many times, and writes the output to a table:

```
repeat 10
reset
run stroop stroop-control 100
reset
run task-switching task-switching 300
run stroop stroop-transfer 100
```

| command | meaning |
|---|---|
| `repeat n` | first line: run the rest of the file n times |
| `reset` | a fresh model |
| `run task label n [params]` | run the task n times. The task is loaded from `task.py` or `task.prims` in the folder of the batch file; the file name and the task name must be the same. The label goes into the output; params are available to the script through `batch-parameters()` |
| `run-time task label seconds [params]` | the same, for a number of simulated seconds |
| `save-image name`, `load-image name` | save or load the state of the model (as Python pickle files) |

Run a batch file in a notebook with `Session().batch("file.bprims", out_path="out.txt", seed=1)`, or from the command line with `python -m pyprims batch file.bprims -o out.txt --seed 1`. `read_batch` (from `pyprims.notebook`) reads the output into a pandas DataFrame with the columns

`repeat task label trial time event p1 p2 p3 in0 in1 in2 in3 in4 firings`

`time` is relative to the start of the trial; `event` is `action`, `trial-end` or `data-line`; `p1`–`p3` hold the action or the trial result; `in0`–`in4` are the fields set with `set-data-file-field`. `plot_batch(data, block=10)` plots the mean trial time per block of trials.

The same can be done in Python, which allows any loop over conditions. `run_trials(model, n, task, events=False, **fields)` (from `pyprims`) is the counterpart of a `run` line: it loads the task, or switches back to it if it is already loaded, runs it `n` times, and returns records for pandas: one per trial end (`trial`, `task`, `result`, `outcome`, any `in0`–`in4` fields, and the `fields` you pass), or with `events=True` one per event, with the columns of the batch output:

```
import random, pandas as pd
from pyprims import Model, run_trials

master, rows = random.Random(1), []
for rep in range(10):
    m = Model(seed=master.random(), trace=False, batch_mode=True)      # reset
    rows += run_trials(m, 100, "count.prims", repeat=rep, label="traincount")
    rows += run_trials(m, 100, "semantic.prims", repeat=rep, label="semantictransfer")
data = pd.DataFrame(rows)
```

## 5. Models in Python

A model can also be written in Python. `python -m pyprims convert model.prims` writes `model.py`, and `session.to_python()` shows the Python form of a loaded model. Operator bodies keep the PRIM notation; the script becomes a Python function, in which every script function is a method of `m`. Python is convenient when the experiment is complicated, or when it reads stimuli from a data file.

In [2]:
from pyprims import Task
from pyprims.notebook import Session

task = Task("count", initial_skills=["count"], default_activation=1.0,
            ol=True, rt=-2.0, lf=0.2, default_operator_self_assoc=0.0,
            egs=0.05, retrieval_reinforces=True)

count = task.skill("count")
count.operator("start-count", """
    V1 <> nil
    WM1 = nil
==>
    V1 -> WM1
    count-fact -> RT1
    V1 -> RT2
    say -> AC1
    V1 -> AC2
""")
count.operator("iterate", """
    RT2 = WM1
    V2 <> WM1
==>
    RT3 -> WM1
    count-fact -> RT1
    RT3 -> RT2
    say -> AC1
    RT3 -> AC2
""")
count.operator("final", """
    V2 = WM1
==>
    say -> AC1
    stop -> AC2
    nil -> G1
""")

digits = ["one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten"]
task.facts([(f"cf{i + 1}", "count-fact", digits[i], digits[i + 1]) for i in range(9)])
task.action("say", latency=0.3, noise=0.1, distribution="uniform", output="Saying")

@task.script
def script(m):
    start = m.random(3)
    end = start + 1 + m.random(3)
    m.print("Counting from", digits[start], "to", digits[end])
    m.screen(digits[start], digits[end])
    m.run_until_action("say", "stop")
    m.issue_reward()
    m.trial_end()

s = Session(seed=1)
s.load(task)
s.run(50)
print(s.trace(1))

Counting from three to five 
289.19  Starting new trial
289.19  *** Retrieved operator start-count with latency 0.009
290.35  *** Retrieved operator iterate with latency 0.001
291.57  *** Retrieved operator iterate with latency 0.003
292.46  *** Retrieved operator final with latency 0.002
293.15  End of trial


A notebook cell that starts with `%%pyprims name` runs Python code that defines a `Task`, and loads it into the session `name`, just as `%%prims name` does for PRIMs syntax.

Without the notebook interface, `pyprims.Model` runs a model directly: `m = Model(seed=1); m.load("count.prims"); m.run(50)`. `m.results` has one value per trial, `m.get_trace(level)` the trace of the last trial, and `m.output_data` the events of the last trial.

## 6. The notebook session

| to … | `Session` | |
|---|---|---|
| load | `s.load(path)` | a `.prims` or `.py` file, a `Task`, or model text; a `%%prims s` or `%%pyprims s` cell. Loading a second task adds it to the model (transfer); loading a task again replaces it and resets the model |
| step | `s.step(n=1)` | one half step: an operator, or the module actions after it |
| | `s.step_operator(n=1)` | until an operator and its module actions are done |
| run | `s.run(n)` | n trials; only the last one is traced |
| | `s.run_time(seconds)` | trials until the model has spent this much model time |
| start again | `s.reset()` | a fresh model with only the current task; results are kept |
| forget everything | `s.clear()` | forget tasks, results and model |
| loaded tasks | `s.tasks()`, `s.select_task(name)` | |
| Trace | `s.trace(level)`, `s.show_trace(level)` | levels 1 operators, 2 + modules, 3 + productions, 4 + compilation, 5 everything; 0 only script output |
| Buffers | `s.buffers()` | |
| Declarative memory | `s.dm(type=None, contains=None)`, `s.chunk(name)` | `dm` is a pandas DataFrame |
| Conflict trace | `s.conflict_set()` | the last retrieval |
| Productions | `s.productions()` | learned productions and their utilities |
| Associations | `s.associations(source=None, type="operator", learned=False)` | $S_{ji}$ from a skill (or other source) to the operators, and how often learning updated them |
| Model results | `s.plot_results(average_window=None, tasks=None, labels=None)`, `s.results_table()` | one line per load or reset of a task; `labels` names the lines |
| Operator activations | `s.track_operator_activations = True`, then `s.plot_operator_activations()` | |
| Task graph | `s.plot_graph(1)`, `s.plot_graph(2)`, `s.plot_graph("productions")`, `s.plot_graph("dm")` | |
| run a batch file | `s.batch(file, out_path=None, seed=None)`; `read_batch`, `plot_batch` | |
| | `s.source()`, `s.to_python()` | the model text, and the model in Python |
| everything at once, with buttons | `Dashboard(s)`, `Dashboard(seed=1)` | `d.refresh()` updates it after code cells changed the session |

After `s.run(n)`, a `NoOperatorWarning` reports trials in which the model stopped because no operator matched (switch it off with `s.warn_failed_trials = False`).

Plots return a matplotlib `Axes`, so you can add titles or lines. `with figure(rows, cols) as axes:` puts several plots in one figure; pass `ax=axes[i]` to each plot function.

Stepping and inspecting never change what the model does: a seeded session gives the same results whether you step through it and inspect it, or just run it.